# Huấn luyện Ultralight cho nhân vật AI (ADR-060)

Notebook chạy **trên Kaggle**, không chạy trên laptop. Nó học một UNet nhỏ (Ultralight-Digital-Human) riêng cho nhân vật
`demo_host_02` từ video huấn luyện 3–5 phút do một model "thầy" tạo. Ultralight chép khẩu hình của thầy, nên thầy đặt mức trần chất lượng. Kết quả là `ultralight.pth`,
file mà fork LiveTalking nạp khi chạy `--model ultralight` trên laptop.

**Quy tắc (CLAUDE.md, ADR-036, ADR-060):**
- Video là gương mặt AI, người không có thật. Không đưa video người thật vào notebook.
- Mỗi nhân vật chỉ dùng cho một khách, một shop. Model học từ nhân vật nào chỉ dùng cho nhân vật đó.
- Model, video và `provenance_train.json` **không đưa vào git**. Tải về laptop, lưu ở `assets/avatars/<CHARACTER_ID>/ultralight/<TEACHER>/`.
- Đây là bước huấn luyện offline. Khi live, hình vẫn sinh realtime (ràng buộc 1).

**Chọn thầy (ADR-064)** bằng một biến `TEACHER` ở ô cấu hình:
- `"musetalk"` (dev, bây giờ): video từ `make_musetalk_video.ipynb`, khoảng 30 phút trên T4 miễn phí; miệng mở ít, hơi mờ.
- `"infinitetalk"` (prod, trước buổi demo): video từ `make_infinitetalk_video.ipynb` với `RUN_NAME = "train_v1"`; miệng tốt
  hơn nhưng bản đủ dài cần GPU thuê (hỏi giá trước).

**Đầu vào:** output của notebook thầy đã chọn (gắn qua *Add Input → Notebook Output*). Notebook tự tìm video của thầy đó trong
`/kaggle/input`. Video phải có tiếng. `run.json` cạnh video được chép thành `teacher_run.json`; nếu nó ghi thầy khác thì
notebook dừng.

**Cách chạy:** Accelerator = GPU T4 (một GPU là đủ), Internet = On, rồi **Save & Run All** (chạy nền, tắt trình duyệt vẫn chạy).
Notebook tự dừng huấn luyện trước giới hạn 12 giờ của Kaggle. Chưa đủ epoch thì gắn output của lần này làm input rồi chạy lại:
notebook tự tìm `last.pt` và học tiếp.

**Kết quả** ở `ultralight_out/<CHARACTER_ID>/<TEACHER>/` (mỗi thầy một thư mục, không ghi đè nhau):
- `ultralight.pth`: model cuối, nạp thẳng vào fork.
- `preview_NNN.jpg`: mỗi hàng là ảnh thật | ảnh model vẽ, cùng 6 khung cố định qua các epoch, để xem miệng nét dần.
- `landmark_models/`: 2 model nhận diện mặt của Ultralight, chép vào `vendor/livetalking/models/` để dựng avatar.
- `loss.png`, `history.json`, `crops_check.jpg`, `train.log`, `preprocess.log`, `provenance_train.json`.
- `last.pt` (để học tiếp) và `epoch_NNN.pth` (mỗi `PTH_EVERY` epoch, để chọn lại nếu epoch cuối kém hơn).

## 1. Cấu hình

Chỉ sửa ô này. Mọi giá trị được ghi vào `provenance_train.json`.

In [ ]:
import time

NOTEBOOK_STARTED = time.monotonic()

CHARACTER_ID = "demo_host_02"  # one character = one customer, one shop
CREATED_BY = "jintran0501"  # who runs this notebook; required in provenance
# Which model made the training video (ADR-064). Ultralight copies the teacher's mouth, so the teacher sets the
# quality ceiling. "musetalk": dev, fast on a free T4. "infinitetalk": prod, better mouth, switch before the demo.
TEACHER = "musetalk"
TEACHERS = {
    "musetalk": {
        "video": f"{CHARACTER_ID}_musetalk_train_v1.mp4",  # make_musetalk_video.ipynb
        "notebook": "jintran0501/ai-host-musetalk-video-demo-host-02",
    },
    "infinitetalk": {
        "video": f"{CHARACTER_ID}_train_v1.mp4",  # make_infinitetalk_video.ipynb with RUN_NAME = "train_v1"
        "notebook": "jintran0501/ai-host-infinitetalk-demo-host-02",
    },
}
if TEACHER not in TEACHERS:
    raise SystemExit(f"TEACHER phải là một trong {sorted(TEACHERS)}, đang là {TEACHER!r}.")
TRAIN_VIDEO_NAME = TEACHERS[TEACHER]["video"]
TRAIN_VIDEO_PATH = ""  # set on Colab or when the file is not under /kaggle/input
# When the input is not attached, fetch the video from the teacher notebook's saved output with kagglehub.
SOURCE_NOTEBOOK = TEACHERS[TEACHER]["notebook"]
RESUME = True  # continue from a last.pt of this notebook found under /kaggle/input

EPOCHS = 200  # upstream default
BATCH_SIZE = 16  # upstream uses 1; 16 keeps the T4 busy. TODO: compare quality against a smaller batch
LR = 1e-3  # upstream default
NUM_WORKERS = 3  # Kaggle gives 4 CPUs
SAVE_EVERY = 5  # preview, history and last.pt
PTH_EVERY = 25  # each epoch_NNN.pth is about 50 MB
BENCH_STEPS = 40
# Kaggle stops a session at 12 h (720 min). Training stops early enough to leave EXPORT_MARGIN_MIN for the checks
# and for Kaggle to save /kaggle/working.
SESSION_BUDGET_MIN = 690
EXPORT_MARGIN_MIN = 20

## 2. Môi trường và mã Ultralight

Mã `anliyuan/Ultralight-Digital-Human` ghim ở commit `ff0454d` (2025-06-17), commit cuối trước lần đổi cấu trúc UNet b2dd34a. Từ
b2dd34a, tên các lớp trong model không còn khớp `avatars/ultralight/unet.py` của fork, nên model học bằng mã mới không nạp được.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

import torch  # preinstalled by Kaggle/Colab with the right CUDA build

ON_KAGGLE = Path("/kaggle").exists()
ON_COLAB = "google.colab" in sys.modules
if not (ON_KAGGLE or ON_COLAB):
    raise SystemExit("Notebook này chỉ chạy trên Kaggle hoặc Colab (ADR-036, ADR-060), không chạy trên laptop.")
PLATFORM = "kaggle" if ON_KAGGLE else "colab"

# Code, frames and HuBERT stay out of /kaggle/working: that folder is the saved output and is capped at 20 GB.
WORK = Path("/tmp/ultralight_work") if ON_KAGGLE else Path("/content/ultralight_work")
ULTRALIGHT_DIR = WORK / "Ultralight-Digital-Human"
DATA_DIR = WORK / "data"
OUT_DIR = (Path("/kaggle/working") if ON_KAGGLE else Path("/content")) / "ultralight_out" / CHARACTER_ID / TEACHER
for folder in (WORK, DATA_DIR, OUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

ULTRALIGHT_REPO = "https://github.com/anliyuan/Ultralight-Digital-Human"
ULTRALIGHT_COMMIT = "ff0454d31059a95ff230cedbd11222f36441555d"


def run(*cmd: str, cwd: Path | None = None) -> str:
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, stdin=subprocess.DEVNULL)
    if result.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd[:4])} lỗi:\n{result.stderr[-3000:]}")
    return result.stdout.strip()


if not (ULTRALIGHT_DIR / ".git").exists():
    run("git", "clone", "-q", ULTRALIGHT_REPO, str(ULTRALIGHT_DIR))
run("git", "checkout", "-q", "--force", ULTRALIGHT_COMMIT, cwd=ULTRALIGHT_DIR)

# Preprocessing imports these; Kaggle and Colab images normally ship them, so pip runs only when one is missing.
missing = []
for module, package in (("librosa", "librosa"), ("soundfile", "soundfile"), ("transformers", "transformers")):
    try:
        __import__(module)
    except ImportError:
        missing.append(package)
if missing:
    run(sys.executable, "-m", "pip", "install", "-q", *missing)
for tool in ("ffmpeg", "ffprobe"):
    if shutil.which(tool) is None:
        raise SystemExit(f"Máy này không có {tool}.")

GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print(f"Nền tảng: {PLATFORM} · GPU: {GPU_NAME} · torch {torch.__version__} · Ultralight {ULTRALIGHT_COMMIT[:7]}")
if GPU_NAME == "none":
    raise SystemExit("Chưa bật GPU. Kaggle: Accelerator = GPU T4. Colab: Runtime type = T4 GPU.")

## 3. Video huấn luyện

Ultralight với HuBERT cần video 25 khung/giây. Video khác 25 khung/giây được chuyển lại; video đúng 25 thì giữ nguyên, không nén lại.

In [ ]:
from IPython.display import Image, display


def find_video() -> Path:
    if TRAIN_VIDEO_PATH:
        return Path(TRAIN_VIDEO_PATH)
    matches = sorted(Path("/kaggle/input").rglob(TRAIN_VIDEO_NAME)) if ON_KAGGLE else []
    if not matches and ON_KAGGLE and SOURCE_NOTEBOOK:
        # A notebook saved through the API may start without its inputs attached; kagglehub authenticates as the
        # notebook owner inside Kaggle and can download a private notebook's output directly.
        fetch = f"import kagglehub; print(kagglehub.notebook_output_download({SOURCE_NOTEBOOK!r}))"
        try:
            run(sys.executable, "-m", "pip", "install", "-q", "-U", "kagglehub")
            fetched = Path(run(sys.executable, "-c", fetch).splitlines()[-1])
            matches = sorted(fetched.rglob(TRAIN_VIDEO_NAME))
            print(f"Lấy video từ output của {SOURCE_NOTEBOOK}: {fetched}")
        except (RuntimeError, IndexError) as err:
            print(f"kagglehub không lấy được output của {SOURCE_NOTEBOOK}: {err}")
    if not matches:
        # Show the videos that are mounted: a wrong name is easier to spot than a missing input.
        videos = sorted(str(p) for p in Path("/kaggle/input").rglob("*.mp4"))[:40] if ON_KAGGLE else []
        raise SystemExit(
            f"Không thấy {TRAIN_VIDEO_NAME} trong /kaggle/input (video đang có: {videos}). Gắn output của notebook "
            f"{SOURCE_NOTEBOOK} (Add Input → Notebook Output), hoặc đổi TEACHER / TRAIN_VIDEO_PATH."
        )
    return matches[0]


def probe(path: Path) -> dict:
    info = json.loads(
        run("ffprobe", "-v", "error", "-print_format", "json", "-show_streams", "-show_format", str(path))
    )
    video = next(s for s in info["streams"] if s["codec_type"] == "video")
    num, den = (int(x) for x in video["r_frame_rate"].split("/"))
    return {
        "width": video["width"],
        "height": video["height"],
        "fps": num / den,
        "duration_s": round(float(info["format"]["duration"]), 2),
        "has_audio": any(s["codec_type"] == "audio" for s in info["streams"]),
    }


video_source = find_video()
# Both teacher notebooks write run.json next to the video (ADR-064); keep it so the model records its teacher.
teacher_run_path = video_source.parent / "run.json"
TEACHER_RUN = json.loads(teacher_run_path.read_text()) if teacher_run_path.exists() else {}
if TEACHER_RUN.get("teacher") not in (None, TEACHER):
    raise SystemExit(f"{teacher_run_path} ghi thầy {TEACHER_RUN['teacher']!r}, nhưng TEACHER = {TEACHER!r}.")
if TEACHER_RUN:
    shutil.copy(teacher_run_path, OUT_DIR / "teacher_run.json")
else:
    print(f"Cảnh báo: không có run.json cạnh {video_source.name}; provenance sẽ thiếu thông tin thầy.")
VIDEO_SHA256 = hashlib.sha256(video_source.read_bytes()).hexdigest()
VIDEO_INFO = probe(video_source)
print(f"Video: {video_source} · sha256 {VIDEO_SHA256[:12]} · {VIDEO_INFO}")
if not VIDEO_INFO["has_audio"]:
    raise SystemExit("Video không có tiếng. Ultralight học khẩu hình theo tiếng, nên cần video có tiếng.")
if VIDEO_INFO["duration_s"] < 60:
    print("Cảnh báo: video dưới 1 phút. Tác giả Ultralight khuyên 3–5 phút; model sẽ kém với âm chưa gặp.")

# process.py writes every output next to the video, so the video gets a folder of its own.
TRAIN_VIDEO = DATA_DIR / "train.mp4"
if abs(VIDEO_INFO["fps"] - 25) < 1e-6:
    shutil.copy(video_source, TRAIN_VIDEO)
    VIDEO_CONVERTED = False
else:
    print(f"Chuyển {VIDEO_INFO['fps']:.3f} khung/giây sang 25")
    run(
        "ffmpeg",
        "-y",
        "-v",
        "error",
        "-i",
        str(video_source),
        "-r",
        "25",
        "-c:v",
        "libx264",
        "-crf",
        "14",
        "-pix_fmt",
        "yuv420p",
        "-c:a",
        "aac",
        "-b:a",
        "192k",
        str(TRAIN_VIDEO),
    )
    VIDEO_CONVERTED = True

## 4. HuBERT

Cùng model với đường suy luận của fork (`models/hubert-large-ls960-ft`). Revision ghim theo ngày kiểm 2026-11-18; repo không đổi từ
2022-05-24. `hubert.py` của Ultralight tải model không ghim revision, nên notebook sửa nó để nạp bản đã ghim.

In [ ]:
from huggingface_hub import snapshot_download

HUBERT = {
    "repo": "facebook/hubert-large-ls960-ft",
    "revision": "ece5fabbf034c1073acae96d5401b25be96709d8",
    "license": "Apache-2.0",
}
started = time.monotonic()
HUBERT_DIR = snapshot_download(
    HUBERT["repo"],
    revision=HUBERT["revision"],
    # tf_model.h5 is the same weights for TensorFlow; not needed.
    allow_patterns=["*.json", "pytorch_model.bin"],
)
print(f"HuBERT: {HUBERT_DIR} ({time.monotonic() - started:.0f} s)")

hubert_script = ULTRALIGHT_DIR / "data_utils" / "hubert.py"
text = hubert_script.read_text()
if text.count('"facebook/hubert-large-ls960-ft"') != 2:
    raise RuntimeError("hubert.py khác bản đã kiểm ở commit ff0454d; xem lại trước khi sửa.")
hubert_script.write_text(text.replace('"facebook/hubert-large-ls960-ft"', repr(HUBERT_DIR)))

## 5. Tiền xử lý: tách khung, dò điểm mặt, đặc trưng HuBERT

Chạy `data_utils/process.py --asr hubert` của Ultralight. Video 3 phút mất khoảng 10–20 phút. `crops_check.jpg` cho xem vùng miệng
mà model sẽ học: khung vuông là vùng cắt 160×160, hình chữ nhật đỏ là vùng bị che mà model phải vẽ lại.

In [ ]:
import cv2
import numpy as np

preprocess_log = OUT_DIR / "preprocess.log"
frames_dir = DATA_DIR / "full_body_img"
landmarks_dir = DATA_DIR / "landmarks"
audio_features = DATA_DIR / "aud_hu.npy"
if not audio_features.exists():
    started = time.monotonic()
    # process.py starts hubert.py with a bare `python`; this puts the kernel's interpreter first on PATH.
    env = {**os.environ, "PATH": f"{Path(sys.executable).parent}:{os.environ['PATH']}"}
    with preprocess_log.open("w") as log:
        result = subprocess.run(
            [sys.executable, "process.py", str(TRAIN_VIDEO), "--asr", "hubert"],
            cwd=ULTRALIGHT_DIR / "data_utils",
            stdout=log,
            stderr=subprocess.STDOUT,
            stdin=subprocess.DEVNULL,
            env=env,
        )
    PREPROCESS_SECONDS = round(time.monotonic() - started)
    if result.returncode != 0:
        raise RuntimeError("process.py lỗi:\n" + preprocess_log.read_text(errors="replace")[-3000:])
else:
    PREPROCESS_SECONDS = 0

# process.py runs its steps through os.system, which hides failures; check each output instead.
frame_count = len(list(frames_dir.glob("*.jpg")))
landmark_count = len(list(landmarks_dir.glob("*.lms")))
if not audio_features.exists() or frame_count == 0 or landmark_count != frame_count:
    raise RuntimeError(
        f"Tiền xử lý chưa đủ: {frame_count} khung, {landmark_count} điểm mặt, aud_hu.npy "
        f"{'có' if audio_features.exists() else 'không có'}.\n" + preprocess_log.read_text(errors="replace")[-3000:]
    )
audio_rows = np.load(audio_features, mmap_mode="r").shape[0]
FRAMES = {"video": frame_count, "landmarks": landmark_count, "audio": int(audio_rows)}
print(f"Tiền xử lý: {FRAMES} ({PREPROCESS_SECONDS} s)")
if abs(audio_rows - frame_count) > 2:
    print("Cảnh báo: số khung tiếng và khung hình lệch nhau hơn 2; trainer chỉ dùng phần chung.")


def crop_box(lms: np.ndarray) -> tuple[int, int, int, int]:
    # Same formula as ultralight_train.py and the fork's genavatar.py.
    xmin, ymin, xmax = int(lms[1][0]), int(lms[52][1]), int(lms[31][0])
    return xmin, ymin, xmax, ymin + (xmax - xmin)


widths = []
tiles = []
for index in np.linspace(0, frame_count - 1, num=8, dtype=int):
    lms = np.loadtxt(landmarks_dir / f"{index}.lms", dtype=np.float32).astype(np.int32)
    x0, y0, x1, y1 = crop_box(lms)
    widths.append(x1 - x0)
    tile = cv2.resize(cv2.imread(str(frames_dir / f"{index}.jpg"))[y0:y1, x0:x1], (168, 168))[4:164, 4:164].copy()
    cv2.rectangle(tile, (5, 5), (155, 150), (0, 0, 255), 1)
    tiles.append(cv2.resize(tile, (240, 240), interpolation=cv2.INTER_NEAREST))
cv2.imwrite(
    str(OUT_DIR / "crops_check.jpg"), np.concatenate([np.concatenate(tiles[:4], 1), np.concatenate(tiles[4:], 1)])
)
print(f"Bề rộng vùng cắt trên khung gốc: {min(widths)}–{max(widths)} px (dưới 100 px thì miệng sẽ mờ)")
display(Image(filename=str(OUT_DIR / "crops_check.jpg"), width=960))

## 6. Mã huấn luyện

Bản sao của `scripts/avatar_clips/ultralight_train.py` trong repo (tạo lại notebook sau khi sửa file đó). Mã riêng thay cho
`train.py` của Ultralight vì `train.py` không học tiếp được khi phiên Kaggle dừng, không có fp16, và `datasetsss.py` của commit đã
ghim bị lỗi ở `__len__`. Dữ liệu, loss và model giống hệt upstream; chỉ bỏ SyncNet (tác giả nói không thấy khác biệt và đã bỏ ở bản
mới).

In [ ]:
TRAINER_SOURCE = r'''"""Train an Ultralight-Digital-Human UNet that the LiveTalking fork can load (ADR-060).

Runs on Kaggle through `train_ultralight.ipynb`, which embeds a copy of this file (rebuild the notebook after edits).
It also runs on the laptop for a short check on a small dataset.

Why not upstream `train.py`:
- After upstream commit b2dd34a (2026-05-23) the UNet state_dict keys no longer match the fork's
  `avatars/ultralight/unet.py`, so the model class is imported from a checkout of the pinned pre-refactor commit
  (ff0454d), whose hubert UNet is the fork's.
- The `datasetsss.py` of that commit fails in `__len__` (it compares a numpy array), and its `train.py` has no
  resume, no mixed precision and saves only every 5 epochs; a Kaggle session stops at 12 h.

The data pipeline matches the fork's inference exactly: the crop from landmarks 1, 31 and 52 resized to 168x168, the
inner 160x160, the black rectangle (5, 5, 150, 145) over the mouth area, a random frame of the same video as the
reference, and 8 HuBERT frames (16 x 1024 -> 16x32x32) centred on the video frame. Loss: L1 + 0.01 x MSE of VGG19
conv3_3 features, as upstream. SyncNet is left out: upstream removed it, its author saw no visible difference.
"""

from __future__ import annotations

import argparse
import json
import math
import random
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

CROP_SIZE = 168
INNER = slice(4, 164)
MASK_RECT = (5, 5, 150, 145)  # x, y, w, h; the fork masks the same box at inference
AUDIO_FRAMES_EACH_SIDE = 4  # aud_hu.npy has 2 HuBERT frames per video frame, so 8 rows give 16 x 1024
VGG_CONV3_3 = 15  # vgg19.features[:15] ends at conv3_3 (index 14), the upstream perceptual layer


def crop_box(lms: np.ndarray) -> tuple[int, int, int, int]:
    """Lower-face square used by upstream training and by the fork's genavatar.py."""
    xmin = int(lms[1][0])
    ymin = int(lms[52][1])
    xmax = int(lms[31][0])
    return xmin, ymin, xmax, ymin + (xmax - xmin)


class LowerFaceDataset(Dataset):
    """All crops are decoded once into memory: a 3-minute video is about 400 MB at 168x168, and Kaggle has few CPUs."""

    def __init__(self, dataset_dir: Path, limit: int | None = None) -> None:
        frames_dir = dataset_dir / "full_body_img"
        audio = np.load(dataset_dir / "aud_hu.npy").astype(np.float32)  # [N, 2, 1024]
        count = min(len(list(frames_dir.glob("*.jpg"))), audio.shape[0])
        if limit:
            count = min(count, limit)
        if count < 2:
            raise ValueError(f"{dataset_dir}: cần ít nhất 2 khung có tiếng, đang có {count}")
        self.audio = audio
        self.crops = np.empty((count, CROP_SIZE, CROP_SIZE, 3), dtype=np.uint8)
        self.boxes: list[tuple[int, int, int, int]] = []
        for i in range(count):
            lms = np.loadtxt(dataset_dir / "landmarks" / f"{i}.lms", dtype=np.float32).astype(np.int32)
            box = crop_box(lms)
            img = cv2.imread(str(frames_dir / f"{i}.jpg"))
            x0, y0, x1, y1 = box
            region = img[y0:y1, x0:x1]
            if region.size == 0:
                raise ValueError(f"Khung {i}: vùng cắt {box} rỗng, điểm mặt sai")
            self.crops[i] = cv2.resize(region, (CROP_SIZE, CROP_SIZE), interpolation=cv2.INTER_AREA)
            self.boxes.append(box)

    def __len__(self) -> int:
        return len(self.crops)

    def audio_window(self, index: int) -> torch.Tensor:
        left, right = index - AUDIO_FRAMES_EACH_SIDE, index + AUDIO_FRAMES_EACH_SIDE
        window = np.zeros((right - left, *self.audio.shape[1:]), dtype=np.float32)
        lo, hi = max(left, 0), min(right, self.audio.shape[0])
        window[lo - left : hi - left] = self.audio[lo:hi]
        return torch.from_numpy(window.reshape(16, 32, 32))

    def sample(self, index: int, ref_index: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        real = self.crops[index][INNER, INNER].copy()
        masked = cv2.rectangle(real.copy(), MASK_RECT, (0, 0, 0), -1)
        ref = self.crops[ref_index][INNER, INNER]

        def to_tensor(img: np.ndarray) -> torch.Tensor:
            return torch.from_numpy(img.transpose(2, 0, 1).astype(np.float32) / 255.0)

        inputs = torch.cat([to_tensor(ref), to_tensor(masked)], dim=0)
        return inputs, to_tensor(real), self.audio_window(index)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        return self.sample(index, random.randrange(len(self)))


class PerceptualLoss(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        from torchvision.models import VGG19_Weights, vgg19

        # IMAGENET1K_V1 is what upstream's `pretrained=True` loads; frozen because only the UNet is optimised.
        self.features = vgg19(weights=VGG19_Weights.IMAGENET1K_V1).features[:VGG_CONV3_3].eval()
        self.features.requires_grad_(False)
        # GTX 16xx cards return NaN from VGG19 in fp16 (seen on the laptop's 1650 Ti, 2026-11-18) while the UNet itself
        # is fine; on the first non-finite result the VGG drops to fp32 for good and the UNet keeps its fp16 speed-up.
        self.fp16_ok = True

    def _loss(self, pred: torch.Tensor, real: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            target = self.features(real)
        return nn.functional.mse_loss(self.features(pred).float(), target.float())

    def forward(self, pred: torch.Tensor, real: torch.Tensor) -> torch.Tensor:
        if self.fp16_ok:
            loss = self._loss(pred, real)
            if torch.isfinite(loss):
                return loss
            self.fp16_ok = False
            print("VGG19 fp16 ra NaN/inf trên GPU này, chuyển perceptual loss sang fp32", flush=True)
        with torch.autocast(pred.device.type, enabled=False):
            return self._loss(pred.float(), real.float())


def load_model_class(ultralight_dir: Path) -> type[nn.Module]:
    sys.path.insert(0, str(ultralight_dir))
    from unet import Model  # the pinned pre-refactor upstream checkout

    return Model


def save_preview(net: nn.Module, dataset: LowerFaceDataset, path: Path, device: str) -> None:
    """Fixed frames, reference = first frame, so previews of different epochs are comparable.

    Runs in fp32 like the fork's inference, so a preview shows what the live avatar will show.
    """
    picks = np.linspace(0, len(dataset) - 1, num=min(6, len(dataset)), dtype=int)
    batch = [dataset.sample(int(i), 0) for i in picks]
    inputs = torch.stack([b[0] for b in batch]).to(device)
    audio = torch.stack([b[2] for b in batch]).to(device)
    net.eval()
    with torch.no_grad():
        pred = np.nan_to_num(net(inputs, audio).float().cpu().numpy())
    net.train()
    rows = []
    for (_, real, _), out in zip(batch, pred, strict=True):
        real_img = (real.numpy().transpose(1, 2, 0) * 255).astype(np.uint8)
        pred_img = (out.transpose(1, 2, 0) * 255).clip(0, 255).astype(np.uint8)
        rows.append(np.concatenate([real_img, pred_img], axis=1))
    cv2.imwrite(str(path), np.concatenate(rows, axis=0))


def export_state_dict(net: nn.Module, path: Path) -> None:
    """Plain fp32 state_dict: the fork does `Model(6, 'hubert').load_state_dict(torch.load(...))`."""
    torch.save({k: v.detach().float().cpu() for k, v in net.state_dict().items()}, path)


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument("--dataset_dir", type=Path, required=True)
    parser.add_argument("--ultralight_dir", type=Path, required=True, help="checkout of the pinned upstream commit")
    parser.add_argument("--save_dir", type=Path, required=True)
    parser.add_argument("--epochs", type=int, default=200)
    parser.add_argument("--batch_size", type=int, default=16)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--num_workers", type=int, default=2)
    parser.add_argument("--amp", action="store_true", help="fp16 autocast + GradScaler (tensor cores on T4)")
    parser.add_argument("--save_every", type=int, default=5, help="preview, history and last.pt every N epochs")
    parser.add_argument("--pth_every", type=int, default=0, help="keep an epoch_NNN.pth every N epochs (0: save_every)")
    parser.add_argument("--resume", type=Path, default=None, help="a last.pt written by this script")
    parser.add_argument("--max_minutes", type=float, default=0, help="stop before the next epoch would pass this")
    parser.add_argument("--bench_steps", type=int, default=0, help="time this many steps, print img/s, exit")
    parser.add_argument("--limit_frames", type=int, default=0, help="use only the first N frames (quick checks)")
    parser.add_argument("--seed", type=int, default=0)
    args = parser.parse_args()

    started = time.monotonic()
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    amp = args.amp and device == "cuda"
    args.save_dir.mkdir(parents=True, exist_ok=True)

    dataset = LowerFaceDataset(args.dataset_dir, limit=args.limit_frames or None)
    print(f"Dữ liệu: {len(dataset)} khung, nạp trong {time.monotonic() - started:.0f} s", flush=True)
    loader = DataLoader(
        dataset,
        batch_size=args.batch_size,
        shuffle=True,
        drop_last=len(dataset) > args.batch_size,
        num_workers=args.num_workers,
        pin_memory=device == "cuda",
        persistent_workers=args.num_workers > 0,
    )

    net = load_model_class(args.ultralight_dir)(6, "hubert").to(device)
    perceptual = PerceptualLoss().to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=args.lr)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    l1 = nn.L1Loss()

    start_epoch = 0
    history: list[dict] = []
    if args.resume and args.resume.exists():
        state = torch.load(args.resume, map_location=device, weights_only=False)
        net.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])
        start_epoch = state["epoch"] + 1
        history = state.get("history", [])
        print(f"Tiếp tục từ {args.resume}, epoch {start_epoch}", flush=True)

    def step(batch: tuple[torch.Tensor, ...]) -> tuple[float, float]:
        inputs, real, audio = (t.to(device, non_blocking=True) for t in batch)
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            pred = net(inputs, audio)
            pixel = l1(pred.float(), real)
            loss = pixel + 0.01 * perceptual(pred, real)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        return loss.item(), pixel.item()

    net.train()
    if args.bench_steps:
        done, timed_from, images = 0, 0.0, 0
        warmup = min(3, args.bench_steps)
        while done < args.bench_steps + warmup:
            for batch in loader:
                if done == warmup:
                    if device == "cuda":
                        torch.cuda.synchronize()
                    timed_from = time.monotonic()
                step(batch)
                if done >= warmup:
                    images += batch[0].shape[0]
                done += 1
                if done >= args.bench_steps + warmup:
                    break
        if device == "cuda":
            torch.cuda.synchronize()
        seconds = time.monotonic() - timed_from
        peak_gb = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else 0.0
        result = {"amp": amp, "img_per_s": round(images / seconds, 2), "peak_vram_gb": round(peak_gb, 2)}
        print("BENCH " + json.dumps(result), flush=True)
        return

    epoch_seconds: list[float] = []
    for epoch in range(start_epoch, args.epochs):
        epoch_started = time.monotonic()
        losses, pixels = [], []
        for batch in loader:
            loss, pixel = step(batch)
            if not math.isfinite(loss):
                raise RuntimeError(f"Epoch {epoch}: loss = {loss}. Thử chạy lại không có --amp.")
            losses.append(loss)
            pixels.append(pixel)
        seconds = time.monotonic() - epoch_started
        epoch_seconds.append(seconds)
        record = {
            "epoch": epoch,
            "loss": round(float(np.mean(losses)), 5),
            "l1": round(float(np.mean(pixels)), 5),
            "seconds": round(seconds, 1),
            "img_per_s": round(len(losses) * args.batch_size / seconds, 1),
        }
        history.append(record)
        print("EPOCH " + json.dumps(record), flush=True)

        last = epoch == args.epochs - 1
        elapsed_min = (time.monotonic() - started) / 60
        next_epoch_min = max(epoch_seconds[-3:]) / 60
        out_of_time = bool(args.max_minutes) and elapsed_min + next_epoch_min > args.max_minutes
        if (epoch + 1) % args.save_every == 0 or last or out_of_time:
            # Each export is about 50 MB; keeping fewer keeps a 200-epoch Kaggle output small enough to download.
            if (epoch + 1) % (args.pth_every or args.save_every) == 0 or last or out_of_time:
                export_state_dict(net, args.save_dir / f"epoch_{epoch:03d}.pth")
            save_preview(net, dataset, args.save_dir / f"preview_{epoch:03d}.jpg", device)
            # Written last and replaced atomically: a session killed mid-save still leaves a usable last.pt.
            tmp = args.save_dir / "last.pt.tmp"
            state = {
                "epoch": epoch,
                "model": net.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict(),
                "history": history,
                "args": {k: str(v) for k, v in vars(args).items()},
            }
            torch.save(state, tmp)
            tmp.replace(args.save_dir / "last.pt")
            (args.save_dir / "history.json").write_text(json.dumps(history, indent=1))
        if out_of_time and not last:
            print(f"Dừng ở epoch {epoch} vì sắp hết {args.max_minutes:.0f} phút; chạy lại với --resume.", flush=True)
            break

    export_state_dict(net, args.save_dir / "ultralight.pth")
    print(f"Xong: {args.save_dir / 'ultralight.pth'}", flush=True)


if __name__ == "__main__":
    main()
'''

TRAINER_PATH = WORK / "ultralight_train.py"
TRAINER_PATH.write_text(TRAINER_SOURCE)
TRAINER_SHA256 = hashlib.sha256(TRAINER_SOURCE.encode()).hexdigest()
print(f"Đã ghi {TRAINER_PATH} · sha256 {TRAINER_SHA256[:12]}")

## 7. Đo tốc độ: fp32 và fp16 (AMP)

Chạy thử `BENCH_STEPS` bước mỗi kiểu trên 600 khung đầu. Trên laptop (GTX 1650 Ti), fp16 nhanh hơn 5 lần (3,3 → 17,2 ảnh/giây).
T4 có tensor core nên fp16 cũng nhanh hơn nhiều. Notebook chọn fp16 nếu nhanh hơn ít nhất 10%.

In [ ]:
def trainer_args(*extra: str) -> list[str]:
    return [
        sys.executable,
        str(TRAINER_PATH),
        "--dataset_dir",
        str(DATA_DIR),
        "--ultralight_dir",
        str(ULTRALIGHT_DIR),
        "--batch_size",
        str(BATCH_SIZE),
        "--lr",
        str(LR),
        "--num_workers",
        str(NUM_WORKERS),
        *extra,
    ]


def bench(amp: bool) -> dict:
    args = trainer_args("--save_dir", str(WORK / "bench"), "--bench_steps", str(BENCH_STEPS), "--limit_frames", "600")
    output = run(*args, *(["--amp"] if amp else []))
    line = next(line for line in output.splitlines() if line.startswith("BENCH "))
    return json.loads(line.removeprefix("BENCH "))


BENCH = {"fp32": bench(amp=False), "amp": bench(amp=True)}
USE_AMP = BENCH["amp"]["img_per_s"] >= 1.1 * BENCH["fp32"]["img_per_s"]
best = BENCH["amp" if USE_AMP else "fp32"]
epoch_min = min(frame_count, audio_rows) / best["img_per_s"] / 60
print(json.dumps(BENCH))
print(
    f"Dùng {'fp16 (AMP)' if USE_AMP else 'fp32'}: khoảng {epoch_min:.1f} phút mỗi epoch, {EPOCHS} epoch mất "
    f"{EPOCHS * epoch_min / 60:.1f} giờ"
)

## 8. Huấn luyện

Nếu có `last.pt` của lần chạy trước trong `/kaggle/input` (gắn output của chính notebook này) thì học tiếp từ đó. Notebook kiểm video
của lần trước có giống video lần này không: khác video thì không học tiếp, vì model sẽ trộn hai bộ dữ liệu.

In [ ]:
def find_resume() -> tuple[Path | None, dict]:
    if not (RESUME and ON_KAGGLE):
        return None, {}
    for checkpoint in sorted(Path("/kaggle/input").rglob(f"ultralight_out/{CHARACTER_ID}/{TEACHER}/last.pt")):
        previous = checkpoint.parent / "provenance_train.json"
        info = json.loads(previous.read_text()) if previous.exists() else {}
        if info.get("video", {}).get("sha256") not in (None, VIDEO_SHA256):
            raise SystemExit(f"{checkpoint} học từ video khác (sha256 khác). Gỡ input đó hoặc đặt RESUME = False.")
        if info.get("teacher") not in (None, TEACHER):
            raise SystemExit(f"{checkpoint} học từ thầy {info['teacher']!r}, khác TEACHER = {TEACHER!r}.")
        return checkpoint, info
    return None, {}


RESUME_FROM, RESUMED_RUN = find_resume()
max_minutes = SESSION_BUDGET_MIN - (time.monotonic() - NOTEBOOK_STARTED) / 60 - EXPORT_MARGIN_MIN
if max_minutes < 2 * epoch_min:
    raise SystemExit(f"Chỉ còn {max_minutes:.0f} phút, không đủ cho 2 epoch.")
extra = [
    "--save_dir",
    str(OUT_DIR),
    "--epochs",
    str(EPOCHS),
    "--save_every",
    str(SAVE_EVERY),
    "--pth_every",
    str(PTH_EVERY),
    "--max_minutes",
    f"{max_minutes:.0f}",
]
if USE_AMP:
    extra.append("--amp")
if RESUME_FROM:
    extra += ["--resume", str(RESUME_FROM)]
    print(f"Học tiếp từ {RESUME_FROM}")
print(f"Huấn luyện tối đa {max_minutes:.0f} phút")

started = time.monotonic()
with (OUT_DIR / "train.log").open("a") as log:
    process = subprocess.Popen(
        trainer_args(*extra), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, stdin=subprocess.DEVNULL
    )
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    process.wait()
TRAIN_SECONDS = round(time.monotonic() - started)
if process.returncode != 0:
    raise RuntimeError(f"Huấn luyện lỗi (mã {process.returncode}); xem {OUT_DIR / 'train.log'}.")

## 9. Kiểm model, biểu đồ loss, hồ sơ nguồn gốc

`ultralight.pth` phải nạp được vào `unet.Model(6, "hubert")` ở chế độ strict, như fork làm khi live.

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

sys.path.insert(0, str(ULTRALIGHT_DIR))
from unet import Model  # noqa: E402  the pinned upstream class, same hubert UNet as the fork

model_path = OUT_DIR / "ultralight.pth"
net = Model(6, "hubert")
net.load_state_dict(torch.load(model_path, map_location="cpu"), strict=True)
with torch.no_grad():
    sample = net.eval()(torch.zeros(1, 6, 160, 160), torch.zeros(1, 16, 32, 32))
if tuple(sample.shape) != (1, 3, 160, 160) or not torch.isfinite(sample).all():
    raise RuntimeError(f"Model ra {tuple(sample.shape)} hoặc có NaN.")
MODEL_SHA256 = hashlib.sha256(model_path.read_bytes()).hexdigest()

history = json.loads((OUT_DIR / "history.json").read_text())
figure, axis = plt.subplots(figsize=(8, 4))
axis.plot([h["epoch"] for h in history], [h["loss"] for h in history], label="loss (L1 + 0.01 x VGG)")
axis.plot([h["epoch"] for h in history], [h["l1"] for h in history], label="L1")
axis.set_xlabel("epoch")
axis.set_yscale("log")
axis.legend()
figure.savefig(OUT_DIR / "loss.png", dpi=110, bbox_inches="tight")
print(f"Model nạp strict được · sha256 {MODEL_SHA256[:12]} · epoch cuối {history[-1]}")
display(Image(filename=str(OUT_DIR / "loss.png")))
last_preview = sorted(OUT_DIR.glob("preview_*.jpg"))[-1]
print(f"Ảnh xem trước cuối: {last_preview.name} (trái: ảnh thật, phải: model vẽ)")
display(Image(filename=str(last_preview), width=320))

from torchvision.models import VGG19_Weights  # noqa: E402

# The fork's genavatar crops with the same SCRFD + PFLD landmark models as preprocessing; ship them with the model so
# the laptop crops faces exactly like the training data. On the laptop they go to vendor/livetalking/models/.
LANDMARK_FILES = ("scrfd_2.5g_kps.onnx", "checkpoint_epoch_335.pth.tar")
landmark_out = OUT_DIR / "landmark_models"
landmark_out.mkdir(exist_ok=True)
LANDMARK_SHA256 = {}
for name in LANDMARK_FILES:
    shutil.copy(ULTRALIGHT_DIR / "data_utils" / name, landmark_out / name)
    LANDMARK_SHA256[name] = hashlib.sha256((landmark_out / name).read_bytes()).hexdigest()

provenance = {
    "character_id": CHARACTER_ID,
    "created_by": CREATED_BY,
    "created_at": datetime.now(UTC).isoformat(timespec="seconds"),
    "kind": "ultralight_unet_for_ai_generated_fictional_face",
    "adr": "ADR-060, ADR-064",
    "teacher": TEACHER,
    "teacher_run": {
        "file": "teacher_run.json" if TEACHER_RUN else None,
        "adr": TEACHER_RUN.get("adr"),
        "run_name": TEACHER_RUN.get("run_name"),
        "created_utc": TEACHER_RUN.get("created_utc"),
    },
    "platform": PLATFORM,
    "platform_terms_note": "ADR-036: dùng cho giai đoạn demo; đọc lại điều khoản trước khi khách trả tiền",
    "gpu": GPU_NAME,
    "torch": torch.__version__,
    "ultralight": {
        "repo": ULTRALIGHT_REPO,
        "commit": ULTRALIGHT_COMMIT,
        "license": "TODO: repo không có file LICENSE (ADR-060)",
    },
    "hubert": {**HUBERT, "local_dir": HUBERT_DIR},
    "vgg19_weights": {
        "name": "IMAGENET1K_V1",
        "url": VGG19_Weights.IMAGENET1K_V1.url,
        "license": "TODO: kiểm trước M2",
    },
    "trainer": {"source": "scripts/avatar_clips/ultralight_train.py", "sha256": TRAINER_SHA256},
    "video": {
        "name": video_source.name,
        "path": str(video_source),
        "sha256": VIDEO_SHA256,
        **VIDEO_INFO,
        "converted_to_25fps": VIDEO_CONVERTED,
    },
    "frames": FRAMES,
    "settings": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "amp": USE_AMP,
        "save_every": SAVE_EVERY,
        "pth_every": PTH_EVERY,
        "max_minutes": round(max_minutes),
    },
    "bench": BENCH,
    "timings_s": {
        "preprocess": PREPROCESS_SECONDS,
        "train": TRAIN_SECONDS,
        "notebook": round(time.monotonic() - NOTEBOOK_STARTED),
    },
    "resumed_from": str(RESUME_FROM) if RESUME_FROM else None,
    "resumed_run_created_at": RESUMED_RUN.get("created_at"),
    "landmark_models": {"dir": "landmark_models", "source": "upstream data_utils", "sha256": LANDMARK_SHA256},
    "model": {"file": "ultralight.pth", "sha256": MODEL_SHA256, "epochs_done": history[-1]["epoch"] + 1},
    "history": history,
}
(OUT_DIR / "provenance_train.json").write_text(json.dumps(provenance, ensure_ascii=False, indent=2))
total_mb = sum(p.stat().st_size for p in OUT_DIR.rglob("*") if p.is_file()) / 1e6
print(f"Xong: {len(list(OUT_DIR.iterdir()))} file, {total_mb:.0f} MB trong {OUT_DIR}")